In [11]:
import logging
import json
from pathlib import Path
from typing import Dict, Any

import numpy as np
import pandas as pd
from tqdm import tqdm
import openai

# Configuración básica de logging
def configure_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    return logging.getLogger(__name__)

logger = configure_logging()

# Ruta y API key proporcionadas
DATA_PATH = Path(
    '../data/clientes_clasificados_HSBC.csv'
)
import os
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
assert openai.api_key, "Set OPENAI_API_KEY in a .env file (see README)"
MODEL_NAME = 'gpt-4o-mini'

# Columnas esperadas y numéricas
REQUIRED_COLS = [
    'industria', 'ingresos_anuales', 'utilidad_neta', 'ebitda',
    'crecimiento_industria', 'saldo_promedio_mensual',
    'saldo_promedio_semestral', 'saldo_promedio_anual',
    'ingresos_cuenta', 'Egresos', 'Deposito cuenta propia',
    'impacto_cluster'
]
NUMERIC_COLS = [col for col in REQUIRED_COLS if col != 'impacto_cluster']

# Pasos del prompt para OpenAI
PROMPT_STEPS = (
    "PIENSA PASO A PASO:\n"
    "1) Identifica las 3 métricas con mayor |z_diff| y explica por qué importan.\n"
    "2) Genera un 'perfil ejecutivo' del cluster en un párrafo.\n"
    "3) Lista 3 posibles causas de este comportamiento.\n"
    "4) Sugiere 3 acciones o productos financieros B2B específicos.\n"
    "5) Detecta 2 riesgos o señales de alerta.\n"
    "FORMA DE RESPUESTA:\n"
    "Devuelve un JSON con llaves: profile, drivers, causes, recommendations, risks."
)

def load_and_clean(path: Path) -> pd.DataFrame:
    """Carga el CSV, valida columnas, maneja NAs e imprime logs de validación."""
    logger.info('Leyendo datos desde: %s', path)
    if not path.exists():
        logger.error('No se encontró el archivo en la ruta especificada')
        raise FileNotFoundError(f"Archivo no encontrado: {path}")

    df = pd.read_csv(path)
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        logger.error('Faltan columnas en el CSV: %s', missing)
        raise KeyError(f"Faltan columnas en el CSV: {missing}")

    # Convertir a numérico e imputar medianas si es necesario
    df[NUMERIC_COLS] = df[NUMERIC_COLS].apply(pd.to_numeric, errors='coerce')
    na_counts = df[NUMERIC_COLS].isna().sum()
    if na_counts.any():
        medians = df[NUMERIC_COLS].median()
        df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(medians)
        logger.warning('Se imputaron valores faltantes con medianas: %s', na_counts[na_counts>0].to_dict())

    # Filtrar clusters vacíos
    if df['impacto_cluster'].isna().any():
        count = df['impacto_cluster'].isna().sum()
        df = df.dropna(subset=['impacto_cluster'])
        logger.warning('Se eliminaron %d filas sin cluster asignado', count)

    logger.info('Total filas tras limpieza: %d', len(df))
    logger.info('Clusters únicos detectados: %s', sorted(df['impacto_cluster'].unique()))
    return df


def compute_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula medias, desviaciones, medianas y cuartiles por cluster, y añade z-scores."""
    # Estadísticas globales
    global_means = df[NUMERIC_COLS].mean()
    global_stds = df[NUMERIC_COLS].std().replace(0, 1)

    # Agregaciones por cluster con nombres explícitos
    agg_dict = {}
    for col in NUMERIC_COLS:
        agg_dict[f"{col}_mean"] = (col, 'mean')
        agg_dict[f"{col}_std"] = (col, 'std')
        agg_dict[f"{col}_median"] = (col, 'median')
        agg_dict[f"{col}_q25"] = (col, lambda x: np.percentile(x, 25))
        agg_dict[f"{col}_q75"] = (col, lambda x: np.percentile(x, 75))

    cluster_agg = (
        df
        .groupby('impacto_cluster')
        .agg(**agg_dict)
        .reset_index()
    )
    logger.info('Shape de cluster_stats: %s', cluster_agg.shape)

    # Calcular z-scores personalizados
    for col in NUMERIC_COLS:
        cluster_agg[f"{col}_z"] = (
            cluster_agg[f"{col}_mean"] - global_means[col]
        ) / global_stds[col]

    return cluster_agg


def build_prompt(cluster: Any, row: pd.Series) -> list:
    """Construye los mensajes system, user e instruction para la API de OpenAI."""
    metrics = {
        suffix: {col: float(row[f"{col}_{suffix}"])
                 for col in NUMERIC_COLS}
        for suffix in ['mean', 'std', 'median', 'q25', 'q75', 'z']
    }
    system_msg = {
        'role': 'system',
        'content': (
            'Eres un asistente experto en análisis de clientes B2B para la industria financiera. '
            'Tu tarea es interpretar clusters de empresas a partir de métricas estadísticas. '
            'Debes RESPONDER SÓLO con un JSON válido, sin explicaciones adicionales.'
        )
    }
    user_msg = {
        'role': 'user',
        'content': json.dumps({'cluster': cluster, 'metrics': metrics}, ensure_ascii=False, indent=2)
    }
    instruction_msg = {'role': 'user', 'content': PROMPT_STEPS}
    return [system_msg, user_msg, instruction_msg]


def generate_reports(cluster_stats: pd.DataFrame) -> Dict[Any, str]:
    """Itera por clusters, llama a OpenAI y recopila reporte JSON por cluster."""
    reports = {}
    for _, row in tqdm(cluster_stats.iterrows(), total=len(cluster_stats), desc='Generando interpretaciones'):
        cluster = row['impacto_cluster']
        messages = build_prompt(cluster, row)
        try:
            response = openai.ChatCompletion.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.3,
                max_tokens=1024
            )
            reports[cluster] = response.choices[0].message.content.strip()
        except Exception as e:
            logger.error('Error generando reporte para %s: %s', cluster, e)
            reports[cluster] = json.dumps({'cluster': cluster, 'error': str(e)})
    if not reports:
        logger.warning('No se generaron reportes: cluster_stats vacío')
    return reports


def display_reports(reports: Dict[Any, str]) -> None:
    """Imprime los reportes para cada cluster."""
    for cluster, report in reports.items():
        print(f"\n=== Cluster: {cluster} ===")
        print(report)


if __name__ == '__main__':
    df = load_and_clean(DATA_PATH)
    stats = compute_statistics(df)
    reports = generate_reports(stats)
    display_reports(reports)


2025-06-07 23:03:52 | INFO | Leyendo datos desde: ../data/clientes_clasificados_HSBC.csv
2025-06-07 23:03:52 | WARNING | Se imputaron valores faltantes con medianas: {'industria': 1499}
2025-06-07 23:03:52 | INFO | Total filas tras limpieza: 1499
2025-06-07 23:03:52 | INFO | Clusters únicos detectados: ['Alto', 'Bajo', 'Medio']
2025-06-07 23:03:52 | INFO | Shape de cluster_stats: (3, 56)
Generando interpretaciones: 100%|██████████| 3/3 [00:31<00:00, 10.53s/it]


=== Cluster: Alto ===
```json
{
  "profile": "El cluster 'Alto' se caracteriza por empresas con ingresos anuales promedio de 3.12 millones, una utilidad neta similar, y un EBITDA de 2.70 millones. A pesar de un leve crecimiento negativo en la industria, estas empresas mantienen saldos promedio mensuales y anuales elevados, lo que sugiere una buena gestión financiera. Sin embargo, la alta desviación estándar en egresos y depósitos indica variabilidad en la estabilidad financiera de las empresas dentro del cluster.",
  "drivers": {
    "max_z_diff": {
      "metric": "Deposito cuenta propia",
      "value": 2.860447276346847
    },
    "second_max_z_diff": {
      "metric": "Egresos",
      "value": 1.6330830467730537
    },
    "third_max_z_diff": {
      "metric": "ebitda",
      "value": 1.3889033307138148
    }
  },
  "causes": [
    "Variabilidad en las estrategias de inversión y ahorro de las empresas.",
    "Diferencias en la gestión de costos y gastos operativos entre las empres

In [41]:
import argparse
import json
import logging
import os
import sys
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
from tqdm import tqdm
import openai
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

def configure_logging(level: str = "INFO") -> logging.Logger:
    numeric_level = getattr(logging, level.upper(), logging.INFO)
    logging.basicConfig(
        level=numeric_level,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    return logging.getLogger(__name__)

logger = configure_logging()

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=60),
    retry=retry_if_exception_type(openai.error.OpenAIError)
)
def call_openai(
    messages: List[Dict[str, str]],
    model: str,
    temperature: float,
    max_tokens: int
) -> str:
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

def load_summary(path: Path) -> pd.DataFrame:
    if not path.exists():
        logger.error("Archivo no encontrado: %s", path)
        sys.exit(1)
    df = pd.read_csv(path)
    required = {"industria", "impacto_cluster", "empresas"}
    missing = required - set(df.columns)
    if missing:
        logger.error("Faltan columnas en el CSV: %s", missing)
        sys.exit(1)
    return df

def build_prompt(cluster: Any, distribution: Dict[str, Any], prompt_steps: str) -> List[Dict[str, str]]:
    system_msg = {
        "role": "system",
        "content": (
            "Eres un asistente experto en análisis de clientes B2B para la industria financiera. "
            "Tu tarea es interpretar la distribución de industrias en un cluster. "
            "Responde sólo con un JSON válido, sin texto adicional."
        )
    }
    user_msg = {
        "role": "user",
        "content": json.dumps({"cluster": cluster, "distribution": distribution}, ensure_ascii=False)
    }
    instruction_msg = {"role": "user", "content": prompt_steps}
    return [system_msg, user_msg, instruction_msg]

def generate_reports(
    summary_df: pd.DataFrame,
    prompt_steps: str,
    model: str,
    temperature: float,
    max_tokens: int,
    use_tqdm: bool
) -> Dict[Any, str]:
    pivot = summary_df.pivot_table(
        index="impacto_cluster",
        columns="industria",
        values="empresas",
        fill_value=0
    )
    reports: Dict[Any, str] = {}
    iterator = pivot.iterrows()
    if use_tqdm:
        iterator = tqdm(iterator, total=len(pivot), desc="Generando informes")

    for cluster, row in iterator:
        dist = row.to_dict()
        messages = build_prompt(cluster, dist, prompt_steps)
        try:
            report = call_openai(messages, model, temperature, max_tokens)
        except Exception as e:
            logger.error("Error al generar reporte para %s: %s", cluster, e)
            report = json.dumps({"cluster": cluster, "error": str(e)})
        reports[cluster] = report
    return reports

def print_reports(reports: Dict[Any, str]) -> None:
    """Imprime cada reporte JSON en la salida estándar."""
    for cluster, content in reports.items():
        print(f"\n=== Cluster: {cluster} ===")
        print(content)

def parse_args() -> argparse.Namespace:
    default_input = Path(
        "../data/resumen_clusters_HSBC.csv"
    )
    parser = argparse.ArgumentParser(description="Generar informes JSON para resumen de clusters HSBC.")
    parser.add_argument(
        "--input", "-i", type=Path,
        default=default_input,
        help="Ruta al CSV de resumen de clusters"
    )
    parser.add_argument(
        "--model", default=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        help="Nombre del modelo OpenAI"
    )
    parser.add_argument(
        "--temp", type=float, default=0.3,
        help="Temperatura para la generación"
    )
    parser.add_argument(
        "--max-tokens", type=int, default=1024,
        help="Máximo de tokens por respuesta"
    )
    parser.add_argument(
        "--no-tqdm", action="store_true",
        help="Desactivar barra de progreso"
    )
    args, _ = parser.parse_known_args()
    return args

def main():
    # Configuración inicial de logging
    log_level = os.getenv("LOG_LEVEL", "INFO")
    global logger
    logger = configure_logging(log_level)

    # API key
    load_dotenv()
    api_key_env = os.getenv("OPENAI_API_KEY")
    if api_key_env:
        openai.api_key = api_key_env
    else:
        import os
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
assert openai.api_key, "Set OPENAI_API_KEY in a .env file (see README)"
        logger.warning("Usando API key hardcoded en código")

    args = parse_args()
    df = load_summary(args.input)

    prompt_steps = (
        "PIENSA PASO A PASO:\n"
        "1) Identifica las 3 industrias con mayor representación relativa y explica por qué importan.\n"
        "2) Genera un 'perfil ejecutivo' de este cluster en un párrafo.\n"
        "3) Lista 3 posibles causas de este patrón de distribución.\n"
        "4) Sugiere 3 acciones o productos financieros B2B específicos para este cluster.\n"
        "5) Detecta 2 riesgos o señales de alerta.\n"
        "FORMA DE RESPUESTA:\n"
        "Devuelve un JSON con llaves: profile, drivers, causes, recommendations, risks."
    )

    reports = generate_reports(
        df,
        prompt_steps,
        args.model,
        args.temp,
        args.max_tokens,
        not args.no_tqdm
    )
    print_reports(reports)

if __name__ == "__main__":
    main()

2025-06-07 23:28:23 | WARNING | Usando API key hardcoded en código




Generando informes: 100%|██████████| 3/3 [00:21<00:00,  7.31s/it]


=== Cluster: Alto ===
{
  "profile": "El cluster Alto se caracteriza por una fuerte presencia de industrias como Agricultura, Telecomunicaciones y Construcción, que representan un 13%, 13% y 12% respectivamente. Estas industrias son fundamentales para la economía, ya que la agricultura asegura la producción de alimentos, las telecomunicaciones facilitan la conectividad y la construcción es clave para el desarrollo de infraestructuras. La diversidad en este cluster sugiere un entorno dinámico y en crecimiento, donde los servicios financieros pueden jugar un papel crucial en el apoyo a la innovación y expansión.",
  "drivers": {
    "Agricultura": "Es esencial para la seguridad alimentaria y la sostenibilidad económica.",
    "Telecomunicaciones": "Facilita la comunicación y el acceso a la información, impulsando la digitalización.",
    "Construcción": "Contribuye al desarrollo de infraestructuras necesarias para el crecimiento urbano y rural."
  },
  "causes": [
    "La demanda crecie

In [42]:
import pandas as pd
import openai

# Asegúrate de que openai.api_key ya está configurada en tu entorno o en una celda previa
# Define los loadings en un DataFrame
loadings = pd.DataFrame({
    'PC1': {
        'ingresos_anuales': 0.359444776,
        'utilidad_neta':    0.359444776,
        'ebitda':           0.326317367,
        'crecimiento_industria': 0.013091695,
        'saldo_promedio_mensual': 0.359444776,
        'saldo_promedio_semestral': 0.346527846,
        'saldo_promedio_anual':   0.359444776,
        'ingresos_cuenta':        0.356771165,
        'Egresos':                0.319255075,
        'Deposito cuenta propia': 0.165082949
    },
    'PC2': {
        'ingresos_anuales': 0.007814416,
        'utilidad_neta':    0.007814416,
        'ebitda':          -0.014397379,
        'crecimiento_industria': -0.999102731,
        'saldo_promedio_mensual': 0.007814416,
        'saldo_promedio_semestral': 0.015239139,
        'saldo_promedio_anual':     0.007814416,
        'ingresos_cuenta':          0.009613627,
        'Egresos':                  0.009029147,
        'Deposito cuenta propia':  -0.030594277
    }
})

# Función para pedir la interpretación de cada componente principal
def interpret_pc(pc_name: str, loading_series: pd.Series) -> str:
    prompt = (
        f"Eres un investigador con doctorado en estadística y finanzas. "
        f"Tienes los siguientes loadings para el componente principal {pc_name}:\n"
        f"{loading_series.to_dict()}\n"
        "Interpreta qué dimensión o perfil de cliente B2B captura este componente, "
        "resaltando las variables más importantes y su significado."
    )
    messages = [
        {"role": "system", "content": "Eres un experto en PCA y análisis financiero."},
        {"role": "user",   "content": prompt}
    ]
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.3,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

# Genera una interpretación para cada PC y almacénala en un DataFrame
interpretations = {}
for pc in loadings.columns:
    interpretations[pc] = interpret_pc(pc, loadings[pc])

interpretations_df = pd.DataFrame.from_dict(
    interpretations, orient="index", columns=["Interpretación"]
)

# Muestra el DataFrame con las interpretaciones
display(interpretations_df)

,Interpretación
PC1,"El componente principal PC1, basado en los loa..."
PC2,El análisis de componentes principales (PCA) e...
